Most impoortant Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field
from agents import set_default_openai_client, set_tracing_disabled


load_dotenv(override=True)

google_api_key = ""
openrouter_api_key = ""
groq_api_key = ""


In [47]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [48]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [49]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [50]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="dots-3-note-preview:free", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="llama-3.1-8b-instant", openai_client=groq_client)

In [51]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="openrouter Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="groq Sales Agent",instructions=instructions, model=oss_model)

In [52]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [53]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [54]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [55]:
tools = [tool1, tool2, tool3, send_email_tool]

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""


client = AsyncOpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1",
)

set_default_openai_client(client)


sales_manager = Agent(
    name="Sales Manager",
    instructions=instructions,
    tools=tools,
    model="dots-3-note-preview:free"
)

In [57]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.




I've completed the task successfully:

## Step 1: Generated Drafts
I used two of the three sales_agent tools (sales_agent1 and sales_agent2) to generate different email drafts. The third tool (sales_agent3) was unavailable due to a technical error, but I had two quality drafts to evaluate.

## Step 2: Evaluated and Selected
After reviewing both drafts, I selected the email from **sales_agent2** as the best option because:

- **More personalized approach** - starts by acknowledging the prospect's company growth
- **Strategic questioning** - asks a question that makes prospects think about their pain points
- **Clear pain point identification** - lists specific SOC 2 compliance challenges
- **Strong value proposition** - clearly explains how ComplAI cuts prep time from months to weeks
- **Low-pressure CTA** - offers a specific 15-minute call with a personalized demo
- **Professional formatting** - includes all contact details and makes scheduling easy

## Step 3: Sent the Best Email
I'

[non-fatal] Tracing client error 401. Response data is redacted.


## Part 2: Structured Outputs

Structured Outputs Allow an LLM to generate structured Python objects instead of plain text. This is commonly achieved using JSON Schema + Pydantic, where the LLM generates JSON that is converted into a validated Python object.

In [58]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [59]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [60]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [ ]:
client = AsyncOpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1",
)

set_default_openai_client(client)


checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    output_type=EmailReview,
    model="dots-3-note-preview:free"
)
result = await Runner.run(checker, email)

[non-fatal] Tracing client error 401. Response data is redacted.


[non-fatal] Tracing client error 401. Response data is redacted.


In [64]:
review = result.final_output
print(review)

is_professional=False number_of_sentences=5 contains_placeholders=True


In [65]:
review.is_professional

False

## Part 3: Guardrails 

 guardials provide AI safety and control rules to ensure it can not perform unsafe action.

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

In [73]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [74]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model='dots-3-note-preview:free', output_guardrails=[email_guardrail])

In [75]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

[non-fatal] Tracing client error 401. Response data is redacted.


## On the other hand..

In [76]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

[non-fatal] Tracing client error 401. Response data is redacted.


Howdy partner. If you’re lookin’ to round up more deals and stop your team from gettin’ bucked off by manual workflows, you’ve come to the right watering hole.

At ComplAI, we don't believe in fixin' fences with a toothpick. We help folks like you automate the compliance headache so you can spend your time chasin’ revenue instead of paper trails.

Here’s the pitch that’s been stampeding through my inbox lately:

***

**Subject: Taming the compliance wild west at [Prospect Company Name]**

Howdy [Prospect Name],

Most folks I talk to in [Prospect's Industry] feel like they’re tryin' to herd cats when it comes to keeping their compliance ducks in a row. It’s a messy business that usually ends up slowing down the whole outfit.

I’m reachin’ out because at ComplAI, we’ve built a way to automate those regulatory hurdles, letting your team get back to the real work—closing deals.

I reckon you might be lookin' to tighten up those operations. You got ten minutes next Tuesday to see if we can 

[non-fatal] Tracing client error 401. Response data is redacted.


In [77]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

[non-fatal] Tracing client error 401. Response data is redacted.


The email is not professional or has placeholders and will not be sent


[non-fatal] Tracing client error 401. Response data is redacted.
